# Домашнее задание: рекомендации на основе содержания

Цель работы - построить регрессионную рекомендательную модель для датасета MovieLens. Модель предсказывает оценку пользователя фильму на основе жанров, тегов и статистик по пользователям и фильмам.

Используемые признаки:
- TF-IDF по жанрам фильмов;
- TF-IDF по пользовательским тегам;
- средняя оценка, медиана, дисперсия и количество оценок пользователя;
- средняя оценка, медиана, дисперсия и количество оценок фильма.

Метрика качества: RMSE на тестовой выборке.

In [ ]:
# импорт библиотек

import pandas as pd
import numpy as np
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

## 1. Загрузка и первичный просмотр данных

Данные находятся в архиве MovieLens. После распаковки загружаются три таблицы: оценки (`ratings`), фильмы (`movies`) и пользовательские теги (`tags`).

In [ ]:
# распаковка архива
with zipfile.ZipFile("/content/ml-latest-small(1).zip", "r") as zip_ref:
    zip_ref.extractall("/content/movielens")

# загрузка данных
ratings = pd.read_csv("/content/movielens/ml-latest-small/ratings.csv")
movies = pd.read_csv("/content/movielens/ml-latest-small/movies.csv")
tags = pd.read_csv("/content/movielens/ml-latest-small/tags.csv")

# просмотр данных
display(ratings.head())
display(movies.head())
display(tags.head())

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [ ]:
print("ratings:", ratings.shape)
print("movies:", movies.shape)
print("tags:", tags.shape)

ratings: (100836, 4)
movies: (9742, 3)
tags: (3683, 4)


## 2. Подготовка текстовых признаков

Теги объединяются по каждому фильму в одну строку, чтобы затем применить TF-IDF-векторизацию.

In [ ]:
# объединение всех тегов каждого фильма в одну строку
movie_tags = (
    tags
    .dropna(subset=["tag"])
    .groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.astype(str)))
    .reset_index()
)

movie_tags.head()

,movieId,tag
0,1,pixar pixar fun
1,2,fantasy magic board game Robin Williams game
2,3,moldy old
3,5,pregnancy remake
4,7,remake


In [ ]:
# объединяем оценки, фильмы и теги
df = ratings.merge(movies, on="movieId", how="left")
df = df.merge(movie_tags, on="movieId", how="left")

# заполнение пропусков в текстовых признаках
df["tag"] = df["tag"].fillna("")
df["genres"] = df["genres"].fillna("")

display(df.head())

,userId,movieId,rating,timestamp,title,genres,tag
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance,moldy old
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller,
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,mystery twist ending serial killer
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,mindfuck suspense thriller tricky twist ending...


## 3. Деление на обучающую и тестовую выборки

Сначала данные делятся на train и test. Все статистики по пользователям и фильмам считаются только на обучающей выборке, чтобы избежать утечки данных.

In [ ]:
# test/ train

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print("train:", train_df.shape)
print("test:", test_df.shape)

train: (80668, 7)
test: (20168, 7)


## 4. Статистические признаки пользователя и фильма

Для каждого пользователя и фильма рассчитываются средняя оценка, медиана, дисперсия и количество оценок. Эти признаки помогают модели учитывать общую склонность пользователя ставить высокие/низкие оценки и популярность фильма.

In [ ]:
# статистики по пользователям (по обучающей выборке train)
user_stats = (
    train_df
    .groupby("userId")["rating"]
    .agg(
        user_mean_rating="mean",
        user_median_rating="median",
        user_var_rating="var",
        user_count_rating="count"
    )
    .reset_index()
)

display(user_stats.head())

,userId,user_mean_rating,user_median_rating,user_var_rating,user_count_rating
0,1,4.331606,5.0,0.660298,193
1,2,3.920000,4.0,0.701667,25
2,3,2.580645,2.0,4.518280,31
3,4,3.464706,4.0,1.883362,170
4,5,3.657895,4.0,1.096017,38


In [ ]:
# статистики по фильмам (по обучающей выборке train)
movie_stats = (
    train_df
    .groupby("movieId")["rating"]
    .agg(
        movie_mean_rating="mean",
        movie_median_rating="median",
        movie_var_rating="var",
        movie_count_rating="count"
    )
    .reset_index()
)

display(movie_stats.head())

,movieId,movie_mean_rating,movie_median_rating,movie_var_rating,movie_count_rating
0,1,3.893678,4.0,0.689497,174
1,2,3.373626,3.5,0.786630,91
2,3,3.162500,3.0,1.030609,40
3,4,2.250000,2.5,0.775000,6
4,5,2.955882,3.0,0.929813,34


In [ ]:
# добавляем статистики в train
train_df = train_df.merge(user_stats, on="userId", how="left")
train_df = train_df.merge(movie_stats, on="movieId", how="left")

# добавляем статистики в test
test_df = test_df.merge(user_stats, on="userId", how="left")
test_df = test_df.merge(movie_stats, on="movieId", how="left")

In [ ]:
# заполнение пропусков
global_mean = train_df["rating"].mean()
global_median = train_df["rating"].median()

fill_values = {
    "user_mean_rating": global_mean,
    "user_median_rating": global_median,
    "user_var_rating": 0,
    "user_count_rating": 0,
    "movie_mean_rating": global_mean,
    "movie_median_rating": global_median,
    "movie_var_rating": 0,
    "movie_count_rating": 0
}

train_df = train_df.fillna(fill_values)
test_df = test_df.fillna(fill_values)

## 5. Формирование матрицы признаков

В качестве `X` используются текстовые признаки (`genres`, `tag`) и числовые статистики. Целевая переменная `y` — оценка `rating`.

In [ ]:
# формирование признаков и таргета

features = [
    "genres",
    "tag",
    "user_mean_rating",
    "user_median_rating",
    "user_var_rating",
    "user_count_rating",
    "movie_mean_rating",
    "movie_median_rating",
    "movie_var_rating",
    "movie_count_rating"
]

target = "rating"

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

## 6. Обучение модели

Для жанров и тегов применяется TF-IDF. Числовые признаки масштабируются через `StandardScaler`. В качестве регрессионной модели используется `Ridge Regression`, так как она быстро работает с разреженными TF-IDF-признаками.

In [ ]:
# TF-IDF + регрессионная модель

numeric_features = [
    "user_mean_rating",
    "user_median_rating",
    "user_var_rating",
    "user_count_rating",
    "movie_mean_rating",
    "movie_median_rating",
    "movie_var_rating",
    "movie_count_rating"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "genres_tfidf",
            TfidfVectorizer(token_pattern=r"[^|]+"),
            "genres"
        ),
        (
            "tags_tfidf",
            TfidfVectorizer(max_features=500),
            "tag"
        ),
        (
            "num",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        )
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=1.0))
    ]
)

In [ ]:
# обучение модели
model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('genres_tfidf',
                                                  TfidfVectorizer(token_pattern='[^|]+'),
                                                  'genres'),
                                                 ('tags_tfidf',
                                                  TfidfVectorizer(max_features=500),
                                                  'tag'),
                                                 ('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['user_mean_rating',
                                                   'user_median_rating',
                                                   'user_var_rating',
                                                   'user_count_rating',
                                                   'movie_mean_rating',
                                                   'movie_median_rating',
                                                   'movie_var_rating',
                                                   'movie_count_rating'])])),
                ('regressor', Ridge())])

## 7. Оценка качества модели

Качество модели оценивается с помощью RMSE. Чем ниже RMSE, тем точнее модель предсказывает оценки.

In [ ]:
# предсказание
y_pred = model.predict(X_test)

# RMSE
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print("RMSE модели:", rmse)

RMSE модели: 0.8980031664710693


## 8. Baseline-модель

Для сравнения используется простая baseline-модель, которая всем объектам предсказывает среднюю оценку из обучающей выборки. Если RMSE основной модели ниже baseline, модель действительно улучшает качество предсказания.

In [ ]:
# Baseline для сравнения

baseline_pred = np.full(shape=len(y_test), fill_value=global_mean)
baseline_rmse = mean_squared_error(y_test, baseline_pred) ** 0.5

print("Baseline RMSE:", baseline_rmse)
print("Model RMSE:", rmse)

Baseline RMSE: 1.0488405992661316
Model RMSE: 0.8980031664710693


In [ ]:
if rmse < baseline_rmse:
    print("Модель работает лучше baseline.")
else:
    print("Модель не улучшила baseline.")

Модель работает лучше baseline.


## 9. Пример рекомендаций для пользователя

Для выбранного пользователя берутся фильмы, которые он ещё не оценивал. Модель предсказывает для них оценки, после чего выводятся фильмы с самыми высокими предсказанными оценками.

In [ ]:
# Рекомендации для одного пользователя

user_id = 1

# Фильмы, которые пользователь уже оценивал
rated_movies = ratings[ratings["userId"] == user_id]["movieId"].unique()

# Кандидаты для рекомендации - фильмы, которые пользователь ещё не оценивал
candidate_movies = movies[~movies["movieId"].isin(rated_movies)].copy()

# Добавляем теги
candidate_movies = candidate_movies.merge(movie_tags, on="movieId", how="left")
candidate_movies["tag"] = candidate_movies["tag"].fillna("")
candidate_movies["genres"] = candidate_movies["genres"].fillna("")

# Добавляем userId
candidate_movies["userId"] = user_id

# Добавляем статистики
candidate_movies = candidate_movies.merge(user_stats, on="userId", how="left")
candidate_movies = candidate_movies.merge(movie_stats, on="movieId", how="left")

# Заполняем пропуски
candidate_movies = candidate_movies.fillna(fill_values)

# Предсказываем оценки
X_candidates = candidate_movies[features]
candidate_movies["predicted_rating"] = model.predict(X_candidates)

# Топ-10 рекомендаций
recommendations = (
    candidate_movies
    .sort_values("predicted_rating", ascending=False)
    [["movieId", "title", "genres", "predicted_rating"]]
    .head(10)
)

display(recommendations)

,movieId,title,genres,predicted_rating
9379,176601,Black Mirror,(no genres listed),5.523319
9286,171749,Death Note: Desu nôto (2006–2007),(no genres listed),5.523319
8906,147250,The Adventures of Sherlock Holmes and Doctor W...,(no genres listed),5.523319
9338,174053,Black Mirror: White Christmas (2014),Drama|Horror|Mystery|Sci-Fi|Thriller,5.518973
5004,8580,Into the Woods (1991),Adventure|Comedy|Fantasy|Musical,5.488788
7595,93008,"Very Potter Sequel, A (2010)",Comedy|Musical,5.480936
4741,7614,Oklahoma! (1955),Musical|Romance|Western,5.475633
2126,3086,Babes in Toyland (1934),Children|Comedy|Fantasy|Musical,5.472589
9002,152711,Who Killed Chea Vichea? (2010),Documentary,5.469886
5055,8738,"Woman Is a Woman, A (femme est une femme, Une)...",Comedy|Drama|Musical|Romance,5.468336


## 10. Итоговый вывод

Ниже кратко описаны использованный подход, признаки, модель и полученное качество.

In [ ]:
print(f"""
В данной работе была построена рекомендательная модель на основе содержания для датасета MovieLens.
Целевой переменной являлась оценка пользователя фильму.

В качестве признаков использовались TF-IDF по жанрам и тегам, а также статистики пользователя и фильма:
средняя оценка, медиана, дисперсия и количество оценок.

Статистики были рассчитаны только на обучающей выборке, чтобы избежать утечки данных.
Для обучения использовалась модель Ridge Regression.
Качество модели оценивалось на тестовой выборке с помощью RMSE.

RMSE baseline-модели: {baseline_rmse:.4f}
RMSE построенной модели: {rmse:.4f}
""")


В данной работе была построена рекомендательная модель на основе содержания для датасета MovieLens.
Целевой переменной являлась оценка пользователя фильму.

В качестве признаков использовались TF-IDF по жанрам и тегам, а также статистики пользователя и фильма:
средняя оценка, медиана, дисперсия и количество оценок.

Статистики были рассчитаны только на обучающей выборке, чтобы избежать утечки данных.
Для обучения использовалась модель Ridge Regression.
Качество модели оценивалось на тестовой выборке с помощью RMSE.

RMSE baseline-модели: 1.0488
RMSE построенной модели: 0.8980

